# EDA 09 — Who gets measured directly, and who gets described by inference?

**Question.** EDA 08 showed that an allocation *rate* is definition-dependent, so it cannot carry a demographic comparison. Ask the question in a form that survives that problem: for an individual person, was their record filled in — and does the answer vary by who they are and where they live?

**Plain-English setup.** Every person in PUMS carries six income flags (wages, self-employment, interest, Social Security, retirement, public assistance) marking whether that item was reported or **imputed** by the Bureau. Three facts per person, no denominator required:

| outcome | meaning |
|---|---|
| `any_allocated` | at least one of the six items was filled in |
| `n_items_allocated` | how many, 0–6 |
| `whole_record` | all six — the Bureau substituted a donor record for this person |

These are **per-record properties, not rates.** That is deliberate: EDA 08 showed a rate's denominator is an unstated choice, and none of that applies to "did this specific person's record get filled in."

**The claim being tested.** Characteristic and flag sit on the *same PUMS row*, so "people with less education are more often wholly substituted" is a statement about **people**. Section 5 aggregates to area and the licensed claim changes — that boundary is the **ecological fallacy** and it is flagged explicitly where it bites.

**Reproducibility.** Reads `data/raw/pums_<vintage>_nj_alloc_flags.parquet`. No network, no API key.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook lives in /notebooks; the analysis package lives at the repo root.
REPO_ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from analysis import alloc_denominator as ad
from analysis import alloc_profile as ap
from analysis import common
from analysis import replicate as rep

VINTAGE = common.ACS_VINTAGE
PUMS = REPO_ROOT / "data" / "raw" / f"pums_{VINTAGE}_nj_alloc_flags.parquet"

if not PUMS.exists():
    raise FileNotFoundError(
        f"{PUMS.relative_to(REPO_ROOT)} not found.\n"
        f"Regenerate with:  python ingestion/pull_pums_alloc_flags.py --replicates"
    )

raw = pd.read_parquet(PUMS)
print(f"ACS 5-year PUMS, vintage {VINTAGE}, New Jersey")
print(f"{len(raw):,} person records x {raw.shape[1]} columns")
print(f"replicate weights present: {rep.has_replicates(raw)} "
      f"({len(rep.available_replicates(raw))} of {rep.N_REPLICATES})")


## 1. Universe, and why it must be restricted

Income questions are only asked of people **aged 15 and over**, and educational attainment is only meaningful from **25 and over**. Leaving children in the frame would mix "not asked" with "asked and not answered" — inflating the apparent completeness of the data by counting people who were never in the universe.

`ap.prepare()` applies the income universe and derives the three outcomes.

In [ ]:
prep = ap.prepare(raw)

print(f"raw records            : {len(raw):,}")
print(f"in income universe 15+ : {len(prep):,}  "
      f"({len(prep) / len(raw):.1%})")
print(f"income universe age    : {ap.INCOME_UNIVERSE_AGE}+")
print(f"education universe age : {ap.EDUCATION_UNIVERSE_AGE}+")
print()
print("derived outcomes:", [c for c in prep.columns if c not in raw.columns])

assert prep["AGEP"].min() >= ap.INCOME_UNIVERSE_AGE, "universe must exclude children"
assert prep["n_items_allocated"].between(0, 6).all(), "six sources, so 0-6"
assert (prep["whole_record"] == (prep["n_items_allocated"] == 6)).all(), \
    "whole_record must mean all six"
assert (prep["any_allocated"] == (prep["n_items_allocated"] > 0)).all(), \
    "any_allocated must mean at least one"

print("\nOverall, before splitting by anything:")
w = prep[ap.PERSON_WEIGHT]
for col, label in [("any_allocated", "any item filled in"),
                   ("whole_record", "wholly substituted")]:
    print(f"  {label:<22}: {prep.loc[prep[col], ap.PERSON_WEIGHT].sum() / w.sum():.1%}"
          f" of weighted persons")

# The band constants are (low, high, label) tuples -- the ORDERED label lists
# are what every table and chart below indexes by, so derive them once here.
EDU_LEVELS = [label for _, _, label in ap.EDUCATION_BANDS]
AGE_LEVELS = [label for _, _, label in ap.AGE_BANDS]
SEX_LEVELS = list(ap.SEX_LABELS.values())
LEVELS = {"education": EDU_LEVELS, "age_band": AGE_LEVELS, "sex": SEX_LEVELS}
print(f"\nordered levels: education {len(EDU_LEVELS)}, age {len(AGE_LEVELS)}, "
      f"sex {len(SEX_LEVELS)}")

## 2. The person-level profile

Three characteristics. Education and age are the ones we expect to matter; **sex is included as a control** — if the method manufactured gradients out of nothing, sex would show one too.

In [ ]:
prof = ap.profile_all(prep)
prof.style.format({
    "weighted_persons": "{:,.0f}",
    "share_any_allocated": "{:.1%}",
    "share_whole_record": "{:.1%}",
    "mean_items_allocated": "{:.2f}",
})

In [ ]:
METRIC = "share_whole_record"
spreads = pd.concat([ap.spread(prof, METRIC)]).set_index("characteristic")
print(f"Fold difference between highest and lowest group, {METRIC}\n")
print(spreads[["n_levels", "min", "max", "fold",
               "lowest_level", "highest_level"]].to_string())

edu = prof[prof["characteristic"] == "education"].set_index("level")[METRIC]
age = prof[prof["characteristic"] == "age_band"].set_index("level")[METRIC]

# Headline claim 1: the education gradient is monotonic across ALL levels.
# EDU_LEVELS is ordered least -> most, so the series must be decreasing.
edu_ordered = edu.reindex([b for b in EDU_LEVELS if b in edu.index])
assert edu_ordered.is_monotonic_decreasing, (
    "the education gradient should fall at every step -- if this breaks, the "
    "'perfectly monotonic' claim in WORKLOG.md must be corrected"
)
assert spreads.loc["education", "fold"] > 1.8, "education fold should be near 2x"

# Headline claim 2: sex is a genuine null. This is the internal control.
assert spreads.loc["sex", "fold"] < 1.2, (
    "sex should be nearly flat -- a gradient here would suggest the education "
    "and age patterns are a method artefact rather than a finding"
)

print(f"\neducation: monotonic across all {len(edu_ordered)} levels, "
      f"{spreads.loc['education', 'fold']:.2f}x")
print(f"sex      : {spreads.loc['sex', 'fold']:.2f}x -- a null, and that is the point")

## 3. Are the steps real, or is this noise?

PUMS is a sample, so every share above carries sampling error. Microdata does not ship a margin of error column — instead it ships **80 replicate weights**, each a re-weighting of the sample. Recompute the statistic under all 80 and the spread of those answers *is* the sampling variance:

$$\mathrm{Var}(\theta) = \frac{4}{80}\sum_{r=1}^{80}(\theta_r - \theta)^2 \qquad \mathrm{MOE} = 1.645 \times \mathrm{SE}$$

This is **successive-difference replication**, the Bureau's own method. Add it to [`docs/glossary.md`](../docs/glossary.md).

One subtlety worth stating because getting it wrong is easy: to compare two groups you must form the difference **inside each replicate** and then take the variance. Adding standard errors in quadrature — `sqrt(se_a² + se_b²)` — assumes the two groups are independent samples, and two subgroups of *one* sample are not. `analysis/replicate.py` keeps a deliberately-wrong `naive_difference_se()` so the two can be compared side by side.

In [ ]:
if not rep.has_replicates(raw):
    raise RuntimeError(
        "No replicate weights in this file, so significance cannot be tested.\n"
        "Re-pull with:  python ingestion/pull_pums_alloc_flags.py --replicates"
    )

sig = {}
for char, levels in LEVELS.items():
    present = [l for l in levels if l in set(prep[char].dropna())]
    tbl = rep.pairwise_significance(prep, char, prep["whole_record"],
                                    levels=present)
    # pairwise_significance returns ALL pairs. Adjacency is a property of the
    # ordered level list, so mark it here rather than assuming a column.
    pos = {l: i for i, l in enumerate(present)}
    tbl["adjacent"] = (tbl["level_b"].map(pos) - tbl["level_a"].map(pos)).abs() == 1
    sig[char] = tbl

for char, tbl in sig.items():
    adj = tbl[tbl["adjacent"]]
    print(f"{char:<10} adjacent pairs distinguishable at 90%: "
          f"{int(adj['significant'].sum())} of {len(adj)}"
          f"   (all pairs: {int(tbl['significant'].sum())} of {len(tbl)})")

# The ladders are only a finding if the steps are separable.
for char in ("education", "age_band"):
    adj = sig[char][sig[char]["adjacent"]]
    assert adj["significant"].all(), (
        f"{char}: every adjacent step should clear 90% -- if not, the ladder is "
        f"one flat line drawn with a shaky hand and the write-up must say so"
    )

print("\nMultiple comparisons: with k levels there are k(k-1)/2 tests, so read")
print("the pattern rather than any single row. Adjacent steps are the ones that")
print("carry the 'monotonic ladder' claim.")

In [ ]:
# Why the naive shortcut is wrong, shown rather than asserted.
hi, lo = EDU_LEVELS[0], EDU_LEVELS[-1]          # least vs most educated
ma, mb = (prep["education"] == hi), (prep["education"] == lo)

# Correct: the difference is formed inside each replicate, then varied.
diff = rep.difference(prep, prep["whole_record"], ma, mb)
# Wrong: each group's SE taken separately, then added in quadrature.
sa = rep.share(prep, prep["whole_record"] & ma, ma)
sb = rep.share(prep, prep["whole_record"] & mb, mb)
naive = rep.naive_difference_se(sa["se"], sb["se"])

print(f"{hi} ({sa['estimate']:.1%}) vs {lo} ({sb['estimate']:.1%}), "
      f"whole-record share\n")
print(f"  difference                          : {diff['difference']:+.4f}")
print(f"  SE formed per replicate  (CORRECT)  : {diff['se']:.5f}")
print(f"  SE as sqrt(se_a^2 + se_b^2) (WRONG) : {naive:.5f}")
print(f"  the shortcut is off by {abs(naive - diff['se']) / diff['se']:.0%}")
print(f"\n  significant at 90%: {diff['significant']}  "
      f"(90% CI {diff['ci_low']:+.4f} to {diff['ci_high']:+.4f})")
print("\nThe two groups are subgroups of ONE sample, so their errors are not")
print("independent and quadrature does not apply.")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Chart styling. Single-hue encoding throughout: every chart below shows ONE
# measure, so colour carries magnitude, not identity -- no legend needed and no
# categorical palette to validate. Values are the validated sequential blue ramp.
INK        = "#0b0b0b"
INK_MUTED  = "#52514e"
SURFACE    = "#fcfcfb"
BLUE_400   = "#3987e5"
BLUE_450   = "#2a78d6"
BLUE_RAMP  = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
              "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
              "#184f95", "#104281", "#0d366b"]
SEQ_CMAP   = mpl.colors.LinearSegmentedColormap.from_list("blue_seq", BLUE_RAMP)

mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": INK_MUTED,
    "axes.labelcolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "grid.color": "#e6e5e1",
    "grid.linewidth": 0.8,
    "font.size": 9,
    "figure.dpi": 110,
})

# One measure (whole-record share) across ordered categories, so: single hue,
# no legend (the title names the measure), direct value labels, and 90% error
# bars from the replicate weights. Small multiples rather than a dual axis.
def moe_frame(char, levels):
    t = rep.profile_with_moe(prep, char, prep["whole_record"])
    t = t.set_index("level").reindex([l for l in levels if l in set(t["level"])])
    return t.dropna(subset=["estimate"])


panels = [
    ("education", EDU_LEVELS, "Education"),
    ("age_band", AGE_LEVELS, "Age band"),
    ("sex", SEX_LEVELS, "Sex (control)"),
]

fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.6),
                         gridspec_kw={"width_ratios": [2.1, 2.1, 0.8]})

for ax, (char, levels, title) in zip(axes, panels):
    t = moe_frame(char, levels)
    x = np.arange(len(t))
    ax.bar(x, t["estimate"] * 100, width=0.68, color=BLUE_450,
           yerr=t["moe"] * 100, error_kw={"ecolor": INK_MUTED, "elinewidth": 1.2,
                                          "capsize": 3})
    for xi, v in zip(x, t["estimate"] * 100):
        ax.text(xi, v + 0.9, f"{v:.1f}", ha="center", va="bottom",
                fontsize=8.5, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(t.index, rotation=38, ha="right", fontsize=8)
    ax.set_title(title, loc="left")
    ax.set_ylim(0, max(22, (t["estimate"] * 100 + t["moe"] * 100).max() * 1.18))
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

axes[0].set_ylabel("wholly substituted (%)")
fig.suptitle("Share of people whose entire income record was substituted, "
             "with 90% margins of error",
             x=0.005, ha="left", fontsize=11.5, fontweight="bold")
fig.text(0.005, -0.06,
         "Education and age fall stepwise. Sex is flat -- the control that "
         "shows the other two are not a method artefact.",
         ha="left", fontsize=8.5, color=INK_MUTED)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 4. The 75+ exception, stated rather than smoothed

The age ladder is monotonic until the oldest band, which breaks it — and breaks it in an informative direction. People 75+ are near the **bottom** for whole-record substitution but the **top** for having individual items filled in.

Read plainly: they answer the door, they just do not answer every question. A single pooled "allocation rate" would average those two opposite behaviours into one unremarkable number.

In [ ]:
age_prof = prof[prof["characteristic"] == "age_band"].set_index("level")
cols = ["share_whole_record", "share_any_allocated", "mean_items_allocated"]
view = age_prof.reindex([b for b in AGE_LEVELS if b in age_prof.index])[cols]
print(view.to_string(formatters={
    "share_whole_record": "{:.1%}".format,
    "share_any_allocated": "{:.1%}".format,
    "mean_items_allocated": "{:.2f}".format,
}))

oldest = view.index[-1]
assert view.loc[oldest, "share_any_allocated"] == view["share_any_allocated"].max(), \
    "the oldest band should be highest on any-item allocation"
assert view.loc[oldest, "share_whole_record"] < view["share_whole_record"].max(), \
    "the oldest band should NOT be highest on whole-record substitution"

print(f"\n{oldest}: highest on any-item allocation, "
      f"not highest on whole-record substitution.")
print("Two mechanisms pulling in opposite directions in the same age group.")

## 5. Area level — and the boundary this notebook will not cross

The same outcomes aggregated to **PUMA** (Public Use Microdata Area, the finest geography PUMS publishes — roughly 100,000 people; there is no tract-level microdata).

**This supports a map. It does not support a demographic claim.** "PUMA X has a high rate of substituted records" does not license "people in PUMA X are more likely to have substituted records" — that inference is the **ecological fallacy**, and it is the specific error our own project is about. Section 2 is where person-level claims come from, because there the characteristic and the flag sit on the same row.

In [ ]:
puma_col = "PUMA" if "PUMA" in prep.columns else "public use microdata area"
g = prep.groupby(puma_col)
puma = pd.DataFrame({
    "n_records": g.size(),
    "weighted_persons": g[ap.PERSON_WEIGHT].sum(),
    "share_whole_record": (g.apply(
        lambda d: d.loc[d["whole_record"], ap.PERSON_WEIGHT].sum()
        / d[ap.PERSON_WEIGHT].sum(), include_groups=False)),
}).sort_values("share_whole_record", ascending=False)

fold = puma["share_whole_record"].max() / puma["share_whole_record"].min()
print(f"{len(puma)} PUMAs")
print(f"highest : {puma.index[0]}  {puma['share_whole_record'].iloc[0]:.1%}")
print(f"lowest  : {puma.index[-1]}  {puma['share_whole_record'].iloc[-1]:.1%}")
print(f"fold    : {fold:.2f}x")

edu_fold = spreads.loc["education", "fold"]
print(f"\nGeography ({fold:.2f}x) vs education ({edu_fold:.2f}x): "
      f"place spread is {'wider' if fold > edu_fold else 'narrower'}.")

assert len(puma) > 50, "New Jersey should have on the order of 70+ PUMAs"
print("\nREMINDER: this is an area-level pattern. It licenses a map, not a")
print("statement about the people who live in these areas.")

## 6. What this means

**Established here.** The share of people whose entire income record was substituted falls stepwise as education rises and as age rises, roughly twofold across each ladder, and every adjacent step is distinguishable at 90% using the Bureau's own replicate-weight method. Sex is flat, which is the control that makes the other two credible. Geographic spread across PUMAs is wider than either demographic gradient.

In plain terms: **the less education you have and the younger you are, the more likely it is that "you" in this dataset is a statistical stand-in for you.**

**Two limitations to carry forward.**

*Education and age are correlated, and we did not adjust for one while examining the other.* So these are not two independent findings — they partly describe the same people. A model with both terms would separate them; it has not been run. Until it is, report the gradients as descriptive.

*This is association, not causation.* Nothing here says less education causes nonresponse.

**Open question for the team.** Is this a data-quality finding or a fairness finding? "Less-educated and younger people are more often represented by inference" reads very differently depending on the framing, and the project should choose deliberately rather than let a reader choose for us.

Full write-up: [`docs/allocation-profile-nj.md`](../docs/allocation-profile-nj.md), regenerated by `report_alloc_profile.py`.